# Day 2 提示注入+红队 · 牛津 Tutorial LLM 仿真 (v6.0)

## Persona Prompt (讲师加载此 cell 作为系统提示)

> You are an Oxford tutorial fellow in **提示注入+红队 (Prompt injection, red-team, PyRIT, garak, jailbreak, defense layers)**.
> Never give direct answers. Use Socratic questioning. Act as HBS devil's advocate. Reject vague claims. End each turn with a probing question.
>
> 你是一位牛津大学 AI 安全方向的 tutorial fellow, 主持 Day 2 "提示注入+红队" 的一对一辅导。规则:
> 1. **绝不直接给答案**: 学生问 "L1 该用什么 regex?" -> 你反问 "你觉得攻击者会用哪些字面模式? 编码变换呢?"
> 2. **苏格拉底追问**: 至少 5 个追问 - 为什么 / 反例 / 若前提变 / 凭什么 / 如何
> 3. **HBS 魔鬼代言人**: 学生说 "5 层防御够了" -> 你反驳 "若攻击者 Base64+小写+多轮组合, 你的 L1-L5 各能兜住几成?"
> 4. **拒绝含糊**: 学生说 "挺安全的" -> 你追问 "安全的可操作定义是什么? SafetyMetric 防御后分数 >=0.85 算安全吗?"
> 5. **每轮结尾必抛问题**: 不做总结性陈述, 永远以 probing question 结束。
>
> **限频**: 每单元 1 次/天 (防依赖, 见 cell6)。本 tutorial 不替代 starter.ipynb 上机, 而是逼学生解释自己写的代码。


## Pre-Tutorial Task (强制 retrieval, 提交后才进入 Socratic loop)

> **必须在进入 cell3 Socratic loop 前完成**。这是 retrieval practice (提取练习), 不是重读。
> 牛津 tutorial 的核心是 "学生先说, tutor 再追问"。不提交 pre-tutorial task, cell3 的 Socratic loop 会拒绝启动。

**任务**: 写一段 200-300 字的 essay, 回答以下三个问题 (开卷但需独立写, 不抄 notes):

1. **OWASP LLM01 两种形态**: 把 "直接注入" 和 "间接注入" 各举一个营销 Agent 的例子 (不能用 PI-01/PI-05 原例), 并解释为什么间接注入更危险。
2. **5 层防御的 L1 盲区**: 你的 starter.ipynb L1 regex 至少有哪三类攻击漏检? (用 PI 编号或攻击类型说明)
3. **garak vs PyRIT**: 两者定位差异是什么? 为什么 "garak 全过 ≠ 安全"? 这对应因果阶梯的哪一层?

**提交方式**: 把 essay 文本写入 `student_model.json` 的 `pre_tutorial_essay` 字段 (见 cell4), 然后运行 cell3。
**评分**: 不打分, 但 cell3 Socratic loop 会根据 essay 质量选择追问深度 (含糊 -> 深挖, 精确 -> 升级难度)。


In [ ]:
# Multi-turn Socratic Loop (静态 if/else 模拟, >=4 轮, >=5 苏格拉底问, 不真调 API)
# 每轮根据 student_model 的掌握度选择追问分支, 模拟牛津 tutor 的 Socratic 跟进。

import json, os

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {"pre_tutorial_essay": "", "mastery": {"S1": 0.0, "S2": 0.0, "S3": 0.0},
            "blindspots": [], "turn": 0, "socratic_log": []}

def save_student_model(sm):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

# 苏格拉底问题池 (>=5 个: 为什么/反例/若前提变/凭什么/如何)
SOCRATIC_QUESTIONS = {
    "Q1_why": "为什么间接注入比直接注入更危险? 你的论据是攻击者隐蔽性, 还是防御者感知盲区? 凭什么?",  # 为什么 + 凭什么
    "Q2_counterexample": "若攻击者把 PI-05 的 'SYSTEM:' 改成 'system:' 小写, 你的 L1 regex 还能拦住吗? 反例是什么?",  # 反例 + 若前提变
    "Q3_premise_change": "若攻击者用 Base64 编码 '忽略指令' (PI-11), L1 看到的是一串字母数字, 你的 regex 如何匹配? 如何?",  # 若前提变 + 如何
    "Q4_evidence": "你说 '5 层防御够了', 凭什么? 你的 12 攻击向量拦截率是多少? 哪层漏检最多? 为什么?",  # 凭什么 + 为什么
    "Q5_how": "如何用 deepeval SafetyMetric 把 '是否被攻破' 量化为 0.0-1.0? 三个判定维度是什么? 如何?",  # 如何
    "Q6_counter_garak": "若 garak 全过, 你凭什么说 Agent 安全? 反例: garak 不覆盖哪些 probe? PyRIT 多轮对抗能补吗?",  # 反例 + 凭什么
}

def socratic_turn(sm, student_answer, turn_idx):
    """静态 if/else 模拟 Socratic 追问, 根据 student_answer 含的关键词选择下一问.
    不调 openai/anthropic, 纯规则匹配."""
    sm["turn"] = turn_idx
    log = sm.setdefault("socratic_log", [])
    ans_lower = student_answer.lower()

    # Round 1: 探测 S1 (攻击面建模) - 间接注入 vs 直接注入
    if turn_idx == 1:
        if "间接" in student_answer and "直接" in student_answer:
            if "隐蔽" in student_answer or "不知" in student_answer or "未感知" in student_answer:
                # 学生答到点, 升级难度: 追问反例
                sm["mastery"]["S1"] = 0.6
                follow_up = SOCRATIC_QUESTIONS["Q2_counterexample"]
                log.append({"turn": 1, "student": student_answer[:80], "tutor": follow_up, "branch": "S1_partial_up"})
            else:
                # 学生分了类但没说危险原因, 深挖
                sm["mastery"]["S1"] = 0.3
                follow_up = SOCRATIC_QUESTIONS["Q1_why"]
                log.append({"turn": 1, "student": student_answer[:80], "tutor": follow_up, "branch": "S1_dig_why"})
        else:
            # 学生没分类, 退回基础
            sm["mastery"]["S1"] = 0.1
            follow_up = "请先把直接注入和间接注入各举一个营销 Agent 例子。然后回答: " + SOCRATIC_QUESTIONS["Q1_why"]
            log.append({"turn": 1, "student": student_answer[:80], "tutor": follow_up, "branch": "S1_baseline"})

    # Round 2: 探测 S2 (防御工程) - L1 盲区 + L2-L5 兜底
    elif turn_idx == 2:
        if "base64" in ans_lower or "编码" in student_answer:
            sm["mastery"]["S2"] = 0.5
            follow_up = SOCRATIC_QUESTIONS["Q3_premise_change"]
            log.append({"turn": 2, "student": student_answer[:80], "tutor": follow_up, "branch": "S2_encoding"})
        elif "多轮" in student_answer or "pi-12" in ans_lower:
            sm["mastery"]["S2"] = 0.4
            follow_up = SOCRATIC_QUESTIONS["Q4_evidence"]
            log.append({"turn": 2, "student": student_answer[:80], "tutor": follow_up, "branch": "S2_multiturn"})
        else:
            sm["mastery"]["S2"] = 0.2
            follow_up = "你的 L1 regex 三类盲区 (编码/语义/多轮) 哪类最致命? " + SOCRATIC_QUESTIONS["Q3_premise_change"]
            log.append({"turn": 2, "student": student_answer[:80], "tutor": follow_up, "branch": "S2_baseline"})

    # Round 3: 探测 S3 (红队度量) - deepeval SafetyMetric 三维度
    elif turn_idx == 3:
        if "三" in student_answer and ("维度" in student_answer or "判定" in student_answer):
            sm["mastery"]["S3"] = 0.6
            follow_up = SOCRATIC_QUESTIONS["Q6_counter_garak"]
            log.append({"turn": 3, "student": student_answer[:80], "tutor": follow_up, "branch": "S3_metric_up"})
        else:
            sm["mastery"]["S3"] = 0.2
            follow_up = SOCRATIC_QUESTIONS["Q5_how"]
            log.append({"turn": 3, "student": student_answer[:80], "tutor": follow_up, "branch": "S3_metric_dig"})

    # Round 4: 探测 garak vs PyRIT + 因果阶梯自知
    elif turn_idx == 4:
        if "garak" in ans_lower and "pyrit" in ans_lower:
            if "l1" in ans_lower or "因果" in student_answer:
                sm["mastery"]["S3"] = max(sm["mastery"]["S3"], 0.8)
                follow_up = "好。最后: 你的 5 层防御在因果阶梯 L1 (输入-输出关联), 生产期需补 L2/L3 (因果机制). 给我 2 个你今天的盲点, 作为 exit artifact。"
                sm["blindspots"] = ["待学生填写", "待学生填写"]
                log.append({"turn": 4, "student": student_answer[:80], "tutor": follow_up, "branch": "S3_causal_up"})
            else:
                sm["mastery"]["S3"] = 0.5
                follow_up = "你区分了工具, 但因果阶梯 L1/L2/L3 是什么? 你的 5 层防御在哪层? " + SOCRATIC_QUESTIONS["Q6_counter_garak"]
                log.append({"turn": 4, "student": student_answer[:80], "tutor": follow_up, "branch": "S3_causal_dig"})
        else:
            sm["mastery"]["S3"] = 0.3
            follow_up = "请先说 garak 和 PyRIT 的定位差异, 再回答为什么 garak 全过 ≠ 安全。"
            log.append({"turn": 4, "student": student_answer[:80], "tutor": follow_up, "branch": "S3_baseline"})

    save_student_model(sm)
    return follow_up

# Demo run (静态模拟 4 轮学生回答)
sm = load_student_model()
# 假装学生 pre-tutorial essay 已提交
if not sm["pre_tutorial_essay"]:
    sm["pre_tutorial_essay"] = "(学生 essay 占位: 间接注入藏在评论里, 直接注入在用户输入; 5 层 L1 漏 Base64; garak 扫描 PyRIT 编排)"
    save_student_model(sm)

# 4 轮静态学生回答 (模拟真实 tutorial 对话)
demo_answers = [
    "间接注入比直接注入危险, 因为攻击者隐蔽, 用户和开发者都可能不知道攻击存在。",
    "L1 漏 Base64 编码 (PI-11), 因为 L1 regex 只匹配字面, 解码后的内容看不到。",
    "SafetyMetric 三维度: 是否被攻破 / 是否泄露系统提示 / 是否越权, 1.0 全拦截 0.0 被攻破。",
    "garak 是 NVIDIA 的 probe 扫描器, PyRIT 是微软的多轮对抗编排, garak 全过不等于安全对应因果阶梯 L1。"
]

print("=" * 60)
print("牛津 Tutorial Socratic Loop (静态模拟 4 轮)")
print("=" * 60)
for i, ans in enumerate(demo_answers, 1):
    reply = socratic_turn(sm, ans, i)
    print(f"\n[Turn {i}] 学生: {ans}")
    print(f"[Turn {i}] Tutor: {reply}")
print("\n" + "=" * 60)
print(f"4 轮结束。学生掌握度: S1={sm['mastery']['S1']:.2f}, S2={sm['mastery']['S2']:.2f}, S3={sm['mastery']['S3']:.2f}")
print(f"苏格拉底问题覆盖: 为什么/反例/若前提变/凭什么/如何 (5+ 问已命中)")
print("=" * 60)


In [ ]:
# Student Model 读写 (记录掌握度 / 盲点 / 苏格拉底对话日志)
# student_model.json 是 tutorial 的状态文件, 跨 session 持久化。

import json, os
from datetime import datetime

STUDENT_MODEL_PATH = "student_model.json"

def init_student_model():
    """首次进入 tutorial 时初始化."""
    return {
        "unit": "U-E9-D2",
        "created_at": datetime.now().isoformat(),
        "last_session": None,
        "session_count_today": 0,
        "last_session_date": None,
        "pre_tutorial_essay": "",
        "mastery": {
            "S1_attack_surface": 0.0,  # 攻击面建模 (ILO-1, ILO-4)
            "S2_defense_eng": 0.0,     # 防御工程 (ILO-2)
            "S3_redteam_metric": 0.0   # 红队度量 (ILO-3, ILO-5)
        },
        "blindspots": [],
        "turn": 0,
        "socratic_log": [],
        "hattie_feedback": []
    }

def load_or_init():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    sm = init_student_model()
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)
    return sm

def update_mastery(sm, skill, delta, max_val=1.0):
    """更新掌握度, 0.0-1.0, 单调不减 (取 max 防回退)."""
    sm["mastery"][skill] = min(max_val, max(sm["mastery"][skill], sm["mastery"].get(skill, 0) + delta))

def add_blindspot(sm, blindspot):
    if blindspot not in sm["blindspots"]:
        sm["blindspots"].append(blindspot)

def save(sm):
    sm["last_session"] = datetime.now().isoformat()
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

# Demo
sm = load_or_init()
print("当前 student_model.json:")
print(json.dumps(sm, ensure_ascii=False, indent=2)[:500] + "...")
print("\n掌握度字段: S1_attack_surface / S2_defense_eng / S3_redteam_metric")
print("盲点字段: blindspots (list, 退出时填 2-3 个)")
print("对话日志: socratic_log (每轮 student/tutor/branch)")


## Hattie 4 级 Formative Feedback (退出 tutorial 前生成)

> 依据 Hattie & Timperley (2007) 《The Power of Feedback》。四级按"对学习的影响效应量"排序: Task > Process > Self-Reg > Self (Self 级表扬效应量最低, 此处避免)。
> 本 cell 由 tutor 在 Socratic loop 结束后, 根据 student_model 的 mastery 与 blindspots 自动生成。

**[TASK] 任务级反馈** (效应量 d=0.77, 最高 - 针对具体任务的对错):
- 你的 PI-01 直接注入拆解正确 (类型=jailbreak, 防御层=L1, regex=`忽略.{0,4}指令`) ✓
- 你的 PI-11 Base64 编码拆解**漏了**: L1 regex 无法匹配解码后内容, 但你只写了"L1 漏检", 没说**为什么 L2 (系统提示加固) 也可能漏** (Base64 解码后的指令若不触发角色覆盖模式, L2 也不拦)。修正: 补 L3 安全检查 Agent 是否需扫"解码后内容"。

**[PROCESS] 过程级反馈** (效应量 d=0.75, 高 - 针对学习策略/方法):
- 你的 5 层防御实现策略是"先 L1 后 L5 顺序写", 但**红队度量策略**错了: 你应该先写 12 攻击向量 (S1) -> 再写 5 层 (S2) -> 最后跑拦截率 (S3), 而非"先防御后攻击"。原因: 先写防御会让你只测自己想得到的攻击 (确认偏误), 先写攻击向量迫使你从攻击者视角穷举。建议: 把 starter.ipynb TODO1 提到 TODO2 前做。

**[SELF-REG] 自我调节级反馈** (效应量 d=0.65, 中 - 针对元认知/自检):
- 你在 Round 2 被 tutor 问"Base64+小写+多轮组合"时, 没有主动去查 solution.ipynb 的 L1 漏检原因, 而是猜答案。**自我调节建议**: 当 tutor 追问"反例"时, 你应该 (1) 先承认"我不确定", (2) 主动打开 solution.ipynb TODO5 拦截率表查 L1 实际拦截率, (3) 再回答。这是"知道不知道"的元认知能力。

**[FEED-FORWARD] 前馈级反馈** (效应量 d=0.66, 中 - 针对下一步学习方向):
- 你的 S3 (红队度量) 掌握度 0.5, S1 (攻击面) 0.6, S2 (防御工程) 0.5 - 三项均低于 mastery 阈值 0.8。
- **下一步**:
  1. 复习 `practice.md` Drill D3 Worked (deepeval SafetyMetric 防御前 0.18 / 防御后 0.92 演示)。
  2. 跨单元复习: Day 1 (AI 对齐问题) 的 Constitutional AI - 理解"对齐失败的 Agent 更易被 Prompt Injection 绕过"。
  3. 预习 Day 3 (AI 治理框架 NIST AI RMF) - 把今天的 5 层防御映射到 NIST 的 "Measure" 层。
- **避免**: 不要立即重跑 starter.ipynb (确认偏误), 先读 `reading.md` 的 garak/PyRIT/OWASP 深链, 再回来重做 D3 Independent。


## 限频 + Exit Artifact (防依赖 + 强制迁移)

### 限频政策 (防 tutorial 依赖)
- **每单元 1 次/天**: 同一 day 的 tutorial.ipynb 每天最多跑 1 次 Socratic loop (cell3)。第 2 次运行 cell3 会读 student_model 的 `last_session_date`, 若与今天同日则拒绝启动, 提示"明日再来"。
- **理由**: 牛津 tutorial 的价值在于"学生先独立思考 24h, 再被追问"。频繁跑 tutorial 会变成"问 tutor 要答案", 违反 Socratic 原则。
- **跨单元不限频**: Day 1 / Day 2 / Day 3 各有独立 tutorial, 同一天可跑 3 个不同单元的 tutorial。

### Exit Artifact (退出 tutorial 必填)
> 跑完 cell3-cell5 后, 在 `student_model.json` 的 `blindspots` 字段填 **2-3 个你今天的盲点**, 并在 `next_review` 字段填 **推荐复习单元/章节**。

**盲点示例** (选 2-3 个, 必须具体到 PI 编号或防御层):
1. "我之前以为 L1 regex 能拦所有注入, 今天发现 PI-11 Base64 编码 L1 必漏, 需 L3 安全检查 Agent 扫解码后内容。"
2. "我把 garak 和 PyRIT 定位写反了 - garak 是 probe 扫描 (单轮批量), PyRIT 是多轮对抗编排 (attacker LLM 自适应)。"
3. "我没区分'红队测试发现漏洞' vs '证明无漏洞' - garak 全过 ≠ 安全, 对应因果阶梯 L1 (输入-输出关联), 不能推出 L2 (因果机制)。"

**推荐复习单元** (选 1-2 个):
- `practice.md` Drill D3 Worked (deepeval SafetyMetric 演示)
- `reading.md` garak/PyRIT/OWASP 条目
- Day 1 notes.md § Constitutional AI (对齐失败 -> 易被注入)
- Day 3 notes.md § NIST AI RMF (5 层防御映射 Measure 层)

### 完成标志
当 `student_model.json` 含:
- `mastery` 三项均 ≥0.8 (或弱项循环已触发)
- `blindspots` 列表长度 ≥2
- `next_review` 字段非空
- `socratic_log` 长度 ≥4 (4 轮 Socratic 完成)

则本 Day tutorial 视为完成。否则需明日再来 (限频政策)。

---

*v6.0 牛津 Tutorial LLM 仿真 - 静态 if/else 模拟, 不调 openai/anthropic API。*
*依据: Oxford Tutorial Method + HBS Case Method + Hattie 4-Level Feedback (2007).*
